In [1]:
!pip install apify-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 57.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [apify-client]


In [12]:
from google.colab import userdata

apify_token = userdata.get('APIFY_TOKEN')

In [13]:
import hashlib
import io
import json
import os
import pickle
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
from apify_client import ApifyClient
from scipy.stats import loguniform
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, VotingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, HuberRegressor, Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, TargetEncoder
from sklearn.svm import SVR
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

In [14]:
# ==============================================================================
# 1. CONFIGURACIÓN Y CARGA DE DATOS
# ==============================================================================
APIFY_TOKEN = apify_token
KV_STORE_NAME = "inmobify-storage"
DATASET_KEY = "dataset_inmobiliario_completo_rosario.csv"
DIR_MODELOS = "modelos_entrenados"
ARTIFACT_META = "modelos_ranking_meta.json"


def obtener_kv_store():
    if not APIFY_TOKEN:
        return None, None
    client = ApifyClient(APIFY_TOKEN)
    stores = client.key_value_stores().list().items
    store = next((s for s in stores if getattr(s, "name", None) == KV_STORE_NAME), None)
    if not store:
        raise RuntimeError(f"No se encontró el Key-Value Store '{KV_STORE_NAME}'.")
    return client, client.key_value_store(store.id)


def cargar_dataset() -> pd.DataFrame:
    if os.path.exists(DATASET_KEY):
        print(f"[INFO] Cargando dataset local: {DATASET_KEY}", flush=True)
        return pd.read_csv(DATASET_KEY)

    if not APIFY_TOKEN:
        raise RuntimeError("Definí APIFY_TOKEN o colocá 'dataset_inmobiliario_completo_rosario.csv' en esta carpeta.")

    print(f"[INFO] Descargando '{DATASET_KEY}' desde Apify ({KV_STORE_NAME})...", flush=True)
    _, kv = obtener_kv_store()
    rec = kv.get_record(DATASET_KEY)
    val = rec.get("value") if isinstance(rec, dict) else getattr(rec, "value", None)
    text = val.decode("utf-8") if isinstance(val, bytes) else str(val)
    return pd.read_csv(io.StringIO(text))


# ==============================================================================
# 2. WRAPPER DE INFERENCIA QUE GARANTIZA SALIDA DIRECTA EN USD TOTAL Y USD/m²
# ==============================================================================
class TasadorInmobifyWrapper(BaseEstimator, RegressorMixin):
    """
    Envuelve un Pipeline entrenado sobre log1p(precio_m2_usd) y expone:
      - .predict(X) -> devuelve precio_usd total estimado (ya re-escalado por superficie_m2)
      - .predict_m2(X) -> devuelve precio_m2_usd estimado
    """
    def __init__(self, pipeline_m2):
        self.pipeline_m2 = pipeline_m2

    def fit(self, X, y_usd):
        sup = np.clip(X["superficie_m2"].values, 1.0, None)
        y_log_m2 = np.log1p(y_usd / sup)
        self.pipeline_m2.fit(X, y_log_m2)
        return self

    def predict_m2(self, X):
        pred_log_m2 = self.pipeline_m2.predict(X)
        return np.expm1(pred_log_m2)

    def predict(self, X):
        sup = np.clip(X["superficie_m2"].values, 1.0, None)
        return self.predict_m2(X) * sup

In [15]:
# ==============================================================================
# 3. PREPARACIÓN DE FEATURES Y SPLIT ESTRATIFICADO
# ==============================================================================
df = cargar_dataset()
df = df.dropna(subset=["precio_usd", "superficie_m2", "barrio", "tipo_inmueble"]).reset_index(drop=True)
df = df[(df["precio_usd"] > 0) & (df["superficie_m2"] > 0)].reset_index(drop=True)

FEATURES_LINEALES = [
    "log_superficie_m2",
    "superficie_m2",
    "es_gran_superficie",
    "tiene_expensas",
]

FEATURES_NO_LINEALES_NUM = [
    "log_trafico_resenas_200m",
    "densidad_comercial_200m",
    "diversidad_rubros_200m",
    "dist_polo_mas_cercano_m",
]

FEATURES_CATEGORICAS = ["barrio", "tipo_inmueble"]
COLS_NUM = FEATURES_LINEALES + FEATURES_NO_LINEALES_NUM
COLS_ENTRADA = COLS_NUM + FEATURES_CATEGORICAS

y_usd = df["precio_usd"].values
sup_m2 = df["superficie_m2"].values

idx_train, idx_test = train_test_split(
    np.arange(len(df)),
    test_size=0.25,
    random_state=42,
    stratify=df["posicion_mercado_barrio"],
)

X_train = df.iloc[idx_train][COLS_ENTRADA].reset_index(drop=True)
X_test = df.iloc[idx_test][COLS_ENTRADA].reset_index(drop=True)
y_train_usd, y_test_usd = y_usd[idx_train], y_usd[idx_test]
sup_train, sup_test = sup_m2[idx_train], sup_m2[idx_test]


def construir_preprocesador() -> ColumnTransformer:
    """
    Combina:
      - Imputación mediana + RobustScaler en variables numéricas
      - OneHotEncoder con min_frequency=5 (agrupa barrios con pocos avisos)
      - TargetEncoder suavizado con CV interno (captura nivel base de USD/m² por barrio y tipo)
    """
    return ColumnTransformer([
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]), COLS_NUM),
        ("ohe", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OneHotEncoder(min_frequency=5, handle_unknown="ignore", sparse_output=False)),
        ]), FEATURES_CATEGORICAS),
        ("te", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", TargetEncoder(cv=5, smooth="auto", random_state=42)),
        ]), FEATURES_CATEGORICAS),
    ])


# ==============================================================================
# 4. CATÁLOGO DE MODELOS OPTIMIZADOS CON PÉRDIDA ROBUSTA (L1 / HUBER / REGULARIZACIÓN)
# ==============================================================================
DICT_MODELOS = {
    "Ridge": (Ridge(), {"pipeline_m2__model__alpha": loguniform(1e-1, 5e2)}),
    "Lasso": (Lasso(max_iter=5000), {"pipeline_m2__model__alpha": loguniform(1e-4, 5e-1)}),
    "ElasticNet": (ElasticNet(max_iter=5000), {
        "pipeline_m2__model__alpha": loguniform(1e-4, 5e-1),
        "pipeline_m2__model__l1_ratio": [0.2, 0.5, 0.8],
    }),
    "HuberRegressor": (HuberRegressor(max_iter=2000), {
        "pipeline_m2__model__alpha": loguniform(1e-2, 5e1),
        "pipeline_m2__model__epsilon": [1.1, 1.25, 1.35],
    }),
    "SVR (RBF)": (SVR(kernel="rbf"), {
        "pipeline_m2__model__C": loguniform(1e-1, 3e1),
        "pipeline_m2__model__epsilon": [0.05, 0.1, 0.15],
    }),
    "KNN Regressor": (KNeighborsRegressor(), {
        "pipeline_m2__model__n_neighbors": [7, 11, 15, 21],
        "pipeline_m2__model__weights": ["distance"],
        "pipeline_m2__model__p": [1, 2],
    }),
    "Random Forest": (RandomForestRegressor(criterion="absolute_error", random_state=42, n_jobs=1), {
        "pipeline_m2__model__n_estimators": [200, 350],
        "pipeline_m2__model__max_depth": [6, 8, 12],
        "pipeline_m2__model__min_samples_leaf": [3, 5, 8],
        "pipeline_m2__model__max_features": ["sqrt", 0.7],
    }),
    "Gradient Boosting": (GradientBoostingRegressor(loss="absolute_error", random_state=42), {
        "pipeline_m2__model__n_estimators": [150, 250, 350],
        "pipeline_m2__model__learning_rate": [0.02, 0.04, 0.06],
        "pipeline_m2__model__max_depth": [2, 3, 4],
        "pipeline_m2__model__subsample": [0.75, 0.9],
        "pipeline_m2__model__min_samples_leaf": [4, 8],
    }),
    "XGBoost": (XGBRegressor(objective="reg:absoluteerror", random_state=42, verbosity=0, n_jobs=1), {
        "pipeline_m2__model__n_estimators": [150, 250, 350],
        "pipeline_m2__model__learning_rate": [0.02, 0.04, 0.06],
        "pipeline_m2__model__max_depth": [2, 3, 4],
        "pipeline_m2__model__subsample": [0.75, 0.9],
        "pipeline_m2__model__colsample_bytree": [0.7, 0.9],
        "pipeline_m2__model__reg_alpha": [0.1, 1.0, 5.0],
        "pipeline_m2__model__reg_lambda": [1.0, 5.0, 10.0],
    }),
    "LightGBM": (lgb.LGBMRegressor(objective="regression_l1", random_state=42, verbose=-1, n_jobs=1), {
        "pipeline_m2__model__n_estimators": [150, 250, 350],
        "pipeline_m2__model__learning_rate": [0.02, 0.04, 0.06],
        "pipeline_m2__model__num_leaves": [7, 15, 23],
        "pipeline_m2__model__min_child_samples": [8, 15, 25],
        "pipeline_m2__model__subsample_freq": [1],
        "pipeline_m2__model__subsample": [0.8],
        "pipeline_m2__model__colsample_bytree": [0.7, 0.9],
    }),
}

[INFO] Descargando 'dataset_inmobiliario_completo_rosario.csv' desde Apify (inmobify-storage)...


In [16]:
# ==============================================================================
# 5. BÚSQUEDA, VALIDACIÓN CRUZADA 5-FOLD Y ENSAMBLE FINAL
# ==============================================================================
def main():
    print(f"{'=' * 88}\nENTRENAMIENTO OPTIMIZADO (TARGET USD/m² + L1 LOSS + TARGET ENCODING)\n{'=' * 88}", flush=True)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    resultados = []
    mejores_modelos = {}

    for nombre, (estimador, param_dist) in DICT_MODELOS.items():
        print(f"  -> Optimizando {nombre:<20}...", end=" ", flush=True)
        wrapper = TasadorInmobifyWrapper(
            Pipeline([
                ("prep", construir_preprocesador()),
                ("model", estimador),
            ])
        )

        t0 = time.perf_counter()
        search = RandomizedSearchCV(
            wrapper,
            param_distributions=param_dist,
            n_iter=18,
            cv=kf,
            scoring="neg_mean_absolute_percentage_error",
            random_state=42,
            n_jobs=1,
        )
        search.fit(X_train, y_train_usd)
        t_seg = time.perf_counter() - t0
        best_model = search.best_estimator_

        # Validación cruzada 5-Fold estricta
        mape_folds, mae_m2_folds, mae_usd_folds, r2_log_folds = [], [], [], []
        for tr_i, va_i in kf.split(X_train):
            best_model.fit(X_train.iloc[tr_i], y_train_usd[tr_i])
            pred_usd = best_model.predict(X_train.iloc[va_i])
            real_usd = y_train_usd[va_i]
            s_va = sup_train[va_i]

            mape_folds.append(mean_absolute_percentage_error(real_usd, pred_usd) * 100)
            mae_m2_folds.append(mean_absolute_error(real_usd / s_va, pred_usd / s_va))
            mae_usd_folds.append(mean_absolute_error(real_usd, pred_usd))
            r2_log_folds.append(r2_score(np.log1p(real_usd), np.log1p(pred_usd)))

        # Ajuste final en Train completo y evaluación en Test Holdout
        best_model.fit(X_train, y_train_usd)
        test_pred_usd = best_model.predict(X_test)
        test_mape = mean_absolute_percentage_error(y_test_usd, test_pred_usd) * 100
        test_mae_m2 = mean_absolute_error(y_test_usd / sup_test, test_pred_usd / sup_test)
        test_mae_usd = mean_absolute_error(y_test_usd, test_pred_usd)

        mejores_modelos[nombre] = best_model
        print(
            f"CV MAPE: {np.mean(mape_folds):.2f}% | Test MAPE: {test_mape:.2f}% | Test MAE/m²: USD {test_mae_m2:.2f} ({t_seg:.1f}s)",
            flush=True,
        )

        resultados.append({
            "Modelo": nombre,
            "CV_MAPE (%)": round(float(np.mean(mape_folds)), 2),
            "CV_MAE_m2 (USD)": round(float(np.mean(mae_m2_folds)), 2),
            "CV_MAE_Total (USD)": round(float(np.mean(mae_usd_folds)), 1),
            "CV_R2_log": round(float(np.mean(r2_log_folds)), 3),
            "Test_MAPE (%)": round(float(test_mape), 2),
            "Test_MAE_m2 (USD)": round(float(test_mae_m2), 2),
            "Test_MAE_Total (USD)": round(float(test_mae_usd), 1),
            "Tiempo (s)": round(float(t_seg), 2),
        })

    # Agregar Ensamble Híbrido (Top-3 modelos por CV_MAPE)
    df_tmp = pd.DataFrame(resultados).sort_values(by=["CV_MAPE (%)", "CV_MAE_m2 (USD)"]).reset_index(drop=True)
    top3_nombres = df_tmp.loc[:2, "Modelo"].tolist()
    print(f"  -> Construyendo Ensamble Top-3 ({' + '.join(top3_nombres)})...", end=" ", flush=True)

    t0 = time.perf_counter()
    ensamble_pipe = TasadorInmobifyWrapper(
        VotingRegressor(
            estimators=[(n, mejores_modelos[n].pipeline_m2) for n in top3_nombres],
            weights=[0.4, 0.35, 0.25],
        )
    )
    mape_folds, mae_m2_folds, mae_usd_folds, r2_log_folds = [], [], [], []
    for tr_i, va_i in kf.split(X_train):
        ensamble_pipe.fit(X_train.iloc[tr_i], y_train_usd[tr_i])
        pred_usd = ensamble_pipe.predict(X_train.iloc[va_i])
        real_usd = y_train_usd[va_i]
        s_va = sup_train[va_i]

        mape_folds.append(mean_absolute_percentage_error(real_usd, pred_usd) * 100)
        mae_m2_folds.append(mean_absolute_error(real_usd / s_va, pred_usd / s_va))
        mae_usd_folds.append(mean_absolute_error(real_usd, pred_usd))
        r2_log_folds.append(r2_score(np.log1p(real_usd), np.log1p(pred_usd)))

    ensamble_pipe.fit(X_train, y_train_usd)
    test_pred_usd = ensamble_pipe.predict(X_test)
    test_mape = mean_absolute_percentage_error(y_test_usd, test_pred_usd) * 100
    test_mae_m2 = mean_absolute_error(y_test_usd / sup_test, test_pred_usd / sup_test)
    test_mae_usd = mean_absolute_error(y_test_usd, test_pred_usd)
    t_seg = time.perf_counter() - t0

    mejores_modelos["Ensamble Top-3"] = ensamble_pipe
    print(f"CV MAPE: {np.mean(mape_folds):.2f}% | Test MAPE: {test_mape:.2f}% | Test MAE/m²: USD {test_mae_m2:.2f}", flush=True)

    resultados.append({
        "Modelo": "Ensamble Top-3",
        "CV_MAPE (%)": round(float(np.mean(mape_folds)), 2),
        "CV_MAE_m2 (USD)": round(float(np.mean(mae_m2_folds)), 2),
        "CV_MAE_Total (USD)": round(float(np.mean(mae_usd_folds)), 1),
        "CV_R2_log": round(float(np.mean(r2_log_folds)), 3),
        "Test_MAPE (%)": round(float(test_mape), 2),
        "Test_MAE_m2 (USD)": round(float(test_mae_m2), 2),
        "Test_MAE_Total (USD)": round(float(test_mae_usd), 1),
        "Tiempo (s)": round(float(t_seg), 2),
    })

    df_final = pd.DataFrame(resultados).sort_values(by=["CV_MAPE (%)", "CV_MAE_m2 (USD)"]).reset_index(drop=True)
    print("\n--- RANKING FINAL DE MODELOS OPTIMIZADOS ---", flush=True)
    print(df_final.to_string(index=False), flush=True)

    # Serializar catálogo completo en local y Apify Key-Value Store
    os.makedirs(DIR_MODELOS, exist_ok=True)
    catalogo = {}
    _, kv = obtener_kv_store()

    for idx, row in df_final.iterrows():
        nombre = row["Modelo"]
        modelo_obj = mejores_modelos[nombre]
        slug = nombre.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
        archivo_pkl = f"modelo_{slug}.pkl"
        ruta_local = os.path.join(DIR_MODELOS, archivo_pkl)

        model_bytes = pickle.dumps(modelo_obj)
        sha256_hash = hashlib.sha256(model_bytes).hexdigest()

        with open(ruta_local, "wb") as f:
            f.write(model_bytes)

        if kv:
            kv.set_record(archivo_pkl, model_bytes, content_type="application/octet-stream")

        catalogo[nombre] = {
            "ranking": int(idx + 1),
            "archivo_kv": archivo_pkl,
            "sha256": sha256_hash,
            "metricas": row.to_dict(),
        }

    metadata = {
        "modelo_recomendado": df_final.loc[0, "Modelo"],
        "ensamble_componentes": top3_nombres,
        "features_entrada": COLS_ENTRADA,
        "target_interno": "log1p(precio_m2_usd) -> reconstruido a precio_usd",
        "modelos_disponibles": catalogo,
    }

    with open(ARTIFACT_META, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    if kv:
        kv.set_record(ARTIFACT_META, json.dumps(metadata, ensure_ascii=False), content_type="application/json")
        print(f"\n[LISTO] {len(catalogo)} modelos optimizados y '{ARTIFACT_META}' actualizados en '{KV_STORE_NAME}'.", flush=True)
    else:
        print(f"\n[LISTO] {len(catalogo)} modelos optimizados guardados en '{DIR_MODELOS}/'.", flush=True)


if __name__ == "__main__":
    main()

ENTRENAMIENTO OPTIMIZADO (TARGET USD/m² + L1 LOSS + TARGET ENCODING)
  -> Optimizando Ridge               ... CV MAPE: 34.98% | Test MAPE: 38.91% | Test MAE/m²: USD 2.59 (3.1s)
  -> Optimizando Lasso               ... CV MAPE: 35.02% | Test MAPE: 38.72% | Test MAE/m²: USD 2.58 (4.1s)
  -> Optimizando ElasticNet          ... CV MAPE: 34.96% | Test MAPE: 38.16% | Test MAE/m²: USD 2.56 (3.3s)
  -> Optimizando HuberRegressor      ... CV MAPE: 33.99% | Test MAPE: 37.74% | Test MAE/m²: USD 2.60 (21.2s)
  -> Optimizando SVR (RBF)           ... CV MAPE: 35.41% | Test MAPE: 37.36% | Test MAE/m²: USD 2.64 (4.2s)
  -> Optimizando KNN Regressor       ... CV MAPE: 37.03% | Test MAPE: 44.81% | Test MAE/m²: USD 2.97 (1.4s)
  -> Optimizando Random Forest       ... CV MAPE: 35.43% | Test MAPE: 38.19% | Test MAE/m²: USD 2.60 (136.4s)
  -> Optimizando Gradient Boosting   ... CV MAPE: 33.07% | Test MAPE: 37.92% | Test MAE/m²: USD 2.59 (72.6s)
  -> Optimizando XGBoost             ... CV MAPE: 35.37% | Test